In [25]:
import pandas as pd
import os

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

# Key in dataset title

In [ ]:
# Load reference dataset-title lookup
ref = pd.read_parquet('GBIF/datasets_GBIF_OBIS_filtered.parquet', columns=['alternateIdentifier', 'title'])

# Build a unique lookup table
ref = ref.dropna(subset=['alternateIdentifier']).drop_duplicates(subset=['alternateIdentifier'], keep='first')
title_lookup = ref.set_index('alternateIdentifier')['title']

# Fill missing/empty datasetTitle values from the lookup
df = pd.read_parquet('All_data/merged_all_mof.parquet')
title_series = df['datasetTitle']
missing_title_mask = title_series.isna() | title_series.astype('string').str.strip().eq('')
mapped_titles = df.loc[missing_title_mask, 'datasetKey'].map(title_lookup)

filled_count = mapped_titles.notna().sum()
df.loc[missing_title_mask, 'datasetTitle'] = mapped_titles

# Save back to the same file
output_path = 'All_data/merged_all_mof.parquet'
df.to_parquet(output_path, index=False)

# Coverage summary
overall_coverage = (df['datasetTitle'].notna().sum() / len(df) * 100).round(2)

source_summary = (
    df.assign(_has_datasetTitle=df['datasetTitle'].notna())
      .groupby('sourceDatabase', dropna=False)
      .agg(
          total_records=('_has_datasetTitle', 'size'),
          datasetTitle_non_null=('_has_datasetTitle', 'sum')
      )
)
source_summary['datasetTitle_coverage_percent'] = (
    source_summary['datasetTitle_non_null'] / source_summary['total_records'] * 100
).round(2)

print(f"Filled datasetTitle for {filled_count:,} rows")
print(f"Overall datasetTitle coverage: {overall_coverage:.2f}%")
print("\nCoverage by sourceDatabase:")
print(source_summary.reset_index().to_string(index=False))
print(f"\nSaved to: {output_path}")

# Check column coverage

In [ ]:
# Load the parquet file
df = pd.read_parquet('All_data/merged_all_mof.parquet')

# Compute column coverage and dtypes overview
total_rows = len(df)
summary = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'non_null_count': df.notna().sum().values,
})
summary['coverage_percent'] = (summary['non_null_count'] / total_rows * 100).round(2)

# Save to Deduplicate folder
output_dir = 'Deduplicate'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'column_coverage.csv')
summary.to_csv(output_path, index=False)

# Optional: display the overview
print(summary)
print(f'\nSaved to: {output_path}')

                    column    dtype  non_null_count  coverage_percent
0                       ID    Int64               0              0.00
1   acceptedScientificName   string         3951658            100.00
2                  aphiaID    Int64         3951658            100.00
3              areaSampled  float64          369470              9.35
4          areaSampledUnit   string          448741             11.36
..                     ...      ...             ...               ...
64                    Keep      str         3444743             87.17
65         acceptedAphiaID  float64         3951658            100.00
66                isMarine  boolean         3951324             99.99
67              isBrackish  boolean         3184687             80.59
68      trawlingDistance_m  float64            1483              0.04

[69 rows x 4 columns]

Saved to: Deduplicate\column_coverage.csv


# Deduplicate

In [ ]:
# Define deduplication columns
dedup_cols = ['acceptedAphiaID', 'decimalLatitudeStart', 'decimalLongitudeStart', 'year', 'startDayOfYear']

# Define prioritization columns in order
priority_cols = ['areaSampled', 'biomass', 'gear', 'individualCount']

# Count records before deduplication
records_before = len(df)
print(f"Records before deduplication: {records_before:,}")

# Create a sorting key: prioritize records with non-null values in priority columns
# Sort by: not null in priority_cols (False comes before True, so we negate), then keep first
df_sorted = df.copy()
for col in priority_cols:
	df_sorted[f'{col}_has_value'] = df_sorted[col].notna()

# Sort to bring records with values to the top
sort_cols = [f'{col}_has_value' for col in priority_cols]
df_sorted = df_sorted.sort_values(by=sort_cols, ascending=False)

# Drop duplicates keeping first (which now has priority)
df_deduplicated = df_sorted.drop_duplicates(subset=dedup_cols, keep='first')

# Remove temporary columns
df_deduplicated = df_deduplicated.drop(columns=sort_cols)

# Count records after deduplication
records_after = len(df_deduplicated)
records_removed = records_before - records_after

print(f"Records after deduplication: {records_after:,}")
print(f"Records removed: {records_removed:,}")
print(f"Removal rate: {(records_removed/records_before)*100:.2f}%\n")

# Count removals by sourceDatabase
removal_by_source = df.groupby('sourceDatabase').size() - df_deduplicated.groupby('sourceDatabase').size()
removal_stats = pd.DataFrame({
	'source': removal_by_source.index,
	'removed': removal_by_source.values,
	'total_before': df.groupby('sourceDatabase').size().values,
})
removal_stats['removal_percent'] = (removal_stats['removed'] / removal_stats['total_before'] * 100).round(2)
removal_stats = removal_stats.sort_values('removed', ascending=False)

print("Removals by sourceDatabase:")
print(removal_stats.to_string(index=False))

# Save deduplicated dataset
output_path_dedup = os.path.join(output_dir, 'merged_all_deduplicated.parquet')
df_deduplicated.to_parquet(output_path_dedup)
print(f'\nSaved deduplicated data to: {output_path_dedup}')

Records before deduplication: 3,951,658
Records after deduplication: 2,405,185
Records removed: 1,546,473
Removal rate: 39.13%

Removals by sourceDatabase:
     source  removed  total_before  removal_percent
       OBIS   802129       1340477            59.84
       GBIF   682939       2363384            28.90
CRITTERBASE    61402        240561            25.52
       OGSL        3          7236             0.04

Saved deduplicated data to: Deduplicate\merged_all_deduplicated.parquet


In [ ]:
# Find records that share the same aphiaID, latitude, longitude, and year
group_cols = ['acceptedAphiaID', 'decimalLatitudeStart', 'decimalLongitudeStart', 'year']

# Keep all rows that belong to a duplicated group
dup_mask = df.duplicated(subset=group_cols, keep=False)
year_eval = df.loc[dup_mask].copy()

# Add group size for easier review
year_eval['group_size'] = year_eval.groupby(group_cols)[group_cols[0]].transform('size')

# Sort for readability
year_eval = year_eval.sort_values(by=group_cols + ['group_size'])

# Save to Excel if within Excel row limit, otherwise save as parquet
output_dir = 'Deduplicate'
os.makedirs(output_dir, exist_ok=True)

excel_path = os.path.join(output_dir, 'year_evaluation.xlsx')
parquet_path = os.path.join(output_dir, 'year_evaluation.parquet')

excel_row_limit = 1_048_576

if len(year_eval) <= excel_row_limit:
    year_eval.to_excel(excel_path, index=False)
    print(f"Saved {len(year_eval):,} records to: {excel_path}")
else:
    year_eval.to_parquet(parquet_path, index=False)
    print(f"Too large for Excel ({len(year_eval):,} records). Saved to: {parquet_path}")

Too large for Excel (2,323,597 records). Saved to: Deduplicate\year_evaluation.parquet


In [ ]:
# Read existing deduplicated file
df = pd.read_parquet(output_path_dedup)

# New deduplication keys (add datasetTitle)
dedup_subset = ['acceptedAphiaID', 'decimalLatitudeStart', 'decimalLongitudeStart', 'year', 'datasetTitle']

# Prioritization columns already defined as priority_cols
# Count before
records_before = len(df)
print(f"Records before deduplication: {records_before:,}")

# Create flags and sort to prioritise records with non-null priority cols
df_sorted = df.copy()
for col in priority_cols:
    df_sorted[f'{col}_has_value'] = df_sorted[col].notna()

sort_cols = [f'{c}_has_value' for c in priority_cols]
df_sorted = df_sorted.sort_values(by=sort_cols, ascending=False)

# Drop duplicates keeping the first (which has priority)
df_deduplicated2 = df_sorted.drop_duplicates(subset=dedup_subset, keep='first').drop(columns=sort_cols)

# Count after and report
records_after = len(df_deduplicated2)
records_removed = records_before - records_after
print(f"Records after deduplication: {records_after:,}")
print(f"Records removed: {records_removed:,}")
print(f"Removal rate: {(records_removed/records_before)*100:.2f}%\n")

# Removal stats by sourceDatabase
before_counts = df.groupby('sourceDatabase').size()
after_counts = df_deduplicated2.groupby('sourceDatabase').size()
all_sources = before_counts.index.union(after_counts.index)

before_counts = before_counts.reindex(all_sources, fill_value=0)
after_counts = after_counts.reindex(all_sources, fill_value=0)

removed_series = before_counts - after_counts
removal_stats = pd.DataFrame({
    'source': all_sources,
    'removed': removed_series.values,
    'total_before': before_counts.values
})
removal_stats['removal_percent'] = (removal_stats['removed'] / removal_stats['total_before'] * 100).replace([pd.NaT, pd.NA], 0).fillna(0).round(2)
removal_stats = removal_stats.sort_values('removed', ascending=False).reset_index(drop=True)

print("Removals by sourceDatabase:")
print(removal_stats.to_string(index=False))

# Overwrite the same parquet
df_deduplicated2.to_parquet(output_path_dedup, index=False)
print(f"\nSaved deduplicated data to: {output_path_dedup}")

Records before deduplication: 2,405,185
Records after deduplication: 2,373,858
Records removed: 31,327
Removal rate: 1.30%

Removals by sourceDatabase:
     source  removed  total_before  removal_percent
       GBIF    21222       1680445             1.26
       OBIS     8216        538348             1.53
CRITTERBASE     1889        179159             1.05
       OGSL        0          7233             0.00

Saved deduplicated data to: Deduplicate\merged_all_deduplicated.parquet
